# 🤖 Análisis de Patrones y Automatización de Procesos
## Identificación de Patrones, Clasificación y Automatización con 5000 Registros

Este notebook demuestra cómo:
1. Generar datos realistas (5000 líneas × 10 columnas)
2. Persistir en múltiples formatos (CSV, Excel, SQL, PDF)
3. Analizar patrones con IA
4. Clasificar y automatizar procesos
5. Exportar resultados auditables

## 1️⃣ Configuración de Dependencias y Variables de Entorno

In [17]:
import os
import sys
import pandas as pd
import numpy as np
import json
from datetime import datetime, timedelta, timezone
from pathlib import Path

# Instalar dependencias si es necesario
try:
    import pandas as pd
    import openpyxl
    from reportlab.lib.pagesizes import letter
    from reportlab.platypus import SimpleDocTemplate, Table, TableStyle
except ImportError:
    import subprocess
    print("Instalando dependencias...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "pandas", "openpyxl", "reportlab", "python-dotenv"])

# Configuración de reproducibilidad
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

# Cargar variables de entorno
from dotenv import load_dotenv
load_dotenv()

# Variables de entorno necesarias
API_KEYS = {
    'OPENAI_API_KEY': os.getenv('OPENAI_API_KEY', ''),
    'GEMINI_API_KEY': os.getenv('GEMINI_API_KEY', ''),
    'KIMI_API_KEY': os.getenv('KIMI_API_KEY', ''),
}

# Umbrales configurables por entorno para automatización
AUTOMATION_THRESHOLDS = {
    'BATCH_CONFIDENCE': float(os.getenv('BATCH_CONFIDENCE', '0.85')),
    'REVIEW_NOTIFY_CONFIDENCE': float(os.getenv('REVIEW_NOTIFY_CONFIDENCE', '0.70'))
}

# Metadatos base del pipeline para auditoría
PIPELINE_METADATA = {
    'pipeline_version': '1.2',
    'random_seed': RANDOM_SEED,
    'execution_started_at': datetime.now(timezone.utc).isoformat(),
    'python_version': sys.version.split()[0],
    'automation_thresholds': AUTOMATION_THRESHOLDS
}

# Validar API keys sin mostrar valores
print("✓ Configuración de Variables de Entorno:")
for key in API_KEYS:
    value = API_KEYS[key]
    status = "✓ Configurada" if value else "✗ No configurada"
    print(f"  {key}: {status}")

# Mostrar umbrales activos
print("\n✓ Umbrales activos de automatización:")
for k, v in AUTOMATION_THRESHOLDS.items():
    print(f"  {k}: {v}")

# Crear directorios de trabajo
output_dir = Path("./output_data_processing")
output_dir.mkdir(exist_ok=True)
print(f"\n✓ Directorio de trabajo: {output_dir}")
print(f"✓ Semilla global de reproducibilidad: {RANDOM_SEED}")


✓ Configuración de Variables de Entorno:
  OPENAI_API_KEY: ✗ No configurada
  GEMINI_API_KEY: ✗ No configurada
  KIMI_API_KEY: ✗ No configurada

✓ Umbrales activos de automatización:
  BATCH_CONFIDENCE: 0.85
  REVIEW_NOTIFY_CONFIDENCE: 0.7

✓ Directorio de trabajo: output_data_processing
✓ Semilla global de reproducibilidad: 42


## 2️⃣ Generar Dataset Sintético Realista (5000 filas × 10 columnas)

In [2]:
np.random.seed(42)

# Definir columnas y su descripción
print("Generando dataset de 5000 registros con 10 columnas...\n")
print("Columnas a generar:")
columnas_info = {
    "id": "Identificador único del proceso",
    "fecha": "Fecha del procesamiento",
    "tipo_proceso": "Tipo de proceso de negocio",
    "canal": "Canal de entrada (Web, API, Manual, etc)",
    "tiempo_ciclo": "Tiempo en minutos para completar",
    "costo": "Costo en dólares",
    "errores": "Número de errores durante procesamiento",
    "prioridad": "Nivel de prioridad (Baja, Media, Alta, Urgente)",
    "responsable": "Persona responsable",
    "etiqueta_clase": "Clasificación destino (Automático, Manual, Revisión)"
}

for i, (col, desc) in enumerate(columnas_info.items(), 1):
    print(f"  {i:2}. {col:18} - {desc}")

n_registros = 5000

# Definir valores posibles con patrones
tipos_proceso = ['Facturación', 'Pagos', 'Inventario', 'Logística', 'Soporte']
canales = ['Web', 'API', 'Manual', 'Email', 'Sistema']
prioridades = ['Baja', 'Media', 'Alta', 'Urgente']
responsables = ['Juan', 'María', 'Carlos', 'Ana', 'Pedro', 'Sofia'] 
clasificaciones = ['Automático', 'Manual', 'Revisión']

# Generar datos con patrones controlados
data = {
    'id': [f'PROC{i:05d}' for i in range(1, n_registros + 1)],
    'fecha': [datetime(2024, 1, 1) + timedelta(minutes=np.random.randint(0, 10000)) for _ in range(n_registros)],
    'tipo_proceso': np.random.choice(tipos_proceso, n_registros, p=[0.3, 0.25, 0.2, 0.15, 0.1]),
    'canal': np.random.choice(canales, n_registros, p=[0.35, 0.3, 0.2, 0.1, 0.05]),
    'tiempo_ciclo': np.random.exponential(scale=30, size=n_registros),  # Mayoría rápida, algunas lentas
    'costo': np.random.gamma(shape=2, scale=50, size=n_registros),
    'errores': np.random.poisson(lam=0.5, size=n_registros),  # Mayoría sin errores
    'prioridad': np.random.choice(prioridades, n_registros, p=[0.4, 0.35, 0.2, 0.05]),
    'responsable': np.random.choice(responsables, n_registros),
}

# Generar clasificación con reglas (patrón): facilita modelado posterior
def clasificar(row):
    # Regla simple: automático si es rápido, pocos errores y costo bajo
    if row['tiempo_ciclo'] < 20 and row['errores'] == 0 and row['costo'] < 50 and row['canal'] in ['Web', 'API']:
        return 'Automático'
    # Revisión si hay muchos errores
    elif row['errores'] > 2:
        return 'Revisión'
    # Resto: manual
    else:
        return 'Manual'

df = pd.DataFrame(data)
df['etiqueta_clase'] = df.apply(clasificar, axis=1)

# Convertir tipos de datos
df['fecha'] = pd.to_datetime(df['fecha'])
df['tiempo_ciclo'] = df['tiempo_ciclo'].round(1)
df['costo'] = df['costo'].round(2)

print(f"\n✓ Dataset generado: {len(df)} registros × {len(df.columns)} columnas")
print(f"\nPrimeros 5 registros:")
print(df.head())
print(f"\nÚltimos 5 registros:")
print(df.tail())
print(f"\nEstadísticas descriptivas:")
print(df.describe())


Generando dataset de 5000 registros con 10 columnas...

Columnas a generar:
   1. id                 - Identificador único del proceso
   2. fecha              - Fecha del procesamiento
   3. tipo_proceso       - Tipo de proceso de negocio
   4. canal              - Canal de entrada (Web, API, Manual, etc)
   5. tiempo_ciclo       - Tiempo en minutos para completar
   6. costo              - Costo en dólares
   7. errores            - Número de errores durante procesamiento
   8. prioridad          - Nivel de prioridad (Baja, Media, Alta, Urgente)
   9. responsable        - Persona responsable
  10. etiqueta_clase     - Clasificación destino (Automático, Manual, Revisión)

✓ Dataset generado: 5000 registros × 10 columnas

Primeros 5 registros:
          id               fecha tipo_proceso   canal  tiempo_ciclo   costo  \
0  PROC00001 2024-01-06 01:10:00  Facturación   Email          32.6  107.60   
1  PROC00002 2024-01-01 14:20:00        Pagos     API          13.9  185.20   
2  PROC00

## 3️⃣ Persistir Datos en CSV, Excel y SQL

In [3]:
print("Exportando datos a múltiples formatos...\n")

# 1. CSV
csv_path = output_dir / "procesos_5000_registros.csv"
df.to_csv(csv_path, index=False)
print(f"✓ CSV guardado: {csv_path} ({csv_path.stat().st_size / 1024:.1f} KB)")

# 2. Excel
excel_path = output_dir / "procesos_5000_registros.xlsx"
df.to_excel(excel_path, index=False, sheet_name='Datos')
print(f"✓ Excel guardado: {excel_path} ({excel_path.stat().st_size / 1024:.1f} KB)")

# 3. JSON
json_path = output_dir / "procesos_5000_registros.json"
df.to_json(json_path, orient='records', date_format='iso')
print(f"✓ JSON guardado: {json_path} ({json_path.stat().st_size / 1024:.1f} KB)")

# 4. SQL (SQLite para demo)
import sqlite3
db_path = output_dir / "procesos.db"
conn = sqlite3.connect(db_path)
df.to_sql('procesos', conn, if_exists='replace', index=False)
conn.close()
print(f"✓ SQLite guardado: {db_path} ({db_path.stat().st_size / 1024:.1f} KB)")

# Verificar integridad
print(f"\n✓ Verificación de integridad:")
print(f"  - Registros en CSV: {len(pd.read_csv(csv_path))}")
print(f"  - Registros en Excel: {len(pd.read_excel(excel_path))}")
print(f"  - Registros en JSON: {len(json.load(open(json_path)))}")

# Conectar SQLite y verificar
conn = sqlite3.connect(db_path)
cursor = conn.cursor()
cursor.execute("SELECT COUNT(*) FROM procesos")
sql_count = cursor.fetchone()[0]
conn.close()
print(f"  - Registros en SQL: {sql_count}")


Exportando datos a múltiples formatos...

✓ CSV guardado: output_data_processing/procesos_5000_registros.csv (374.4 KB)
✓ Excel guardado: output_data_processing/procesos_5000_registros.xlsx (258.5 KB)
✓ JSON guardado: output_data_processing/procesos_5000_registros.json (1031.9 KB)
✓ SQLite guardado: output_data_processing/procesos.db (444.0 KB)

✓ Verificación de integridad:
  - Registros en CSV: 5000
  - Registros en Excel: 5000
  - Registros en JSON: 5000
  - Registros en SQL: 5000


## 4️⃣ Análisis de Patrones y Distribuciones

In [4]:
print("ANÁLISIS DE PATRONES IDENTIFICADOS\n")
print("="*80)

# 1. Distribución de Clasificación
print("\n1. DISTRIBUCIÓN DE CLASIFICACIÓN (Variable Objetivo):")
print("-"*80)
distrib_clase = df['etiqueta_clase'].value_counts()
for clase, count in distrib_clase.items():
    pct = (count / len(df)) * 100
    bar = '█' * int(pct / 2)
    print(f"  {clase:15} {count:5} registros ({pct:5.1f}%) {bar}")

# 2. Relación entre variables y clasificación
print("\n2. PATRONES POR TIPO DE PROCESO:")
print("-"*80)
patrones = pd.crosstab(df['tipo_proceso'], df['etiqueta_clase'], margins=True)
print(patrones)

# 3. Análisis por canal
print("\n3. PATRONES POR CANAL:")
print("-"*80)
patrones_canal = pd.crosstab(df['canal'], df['etiqueta_clase'])
print(patrones_canal)

# 4. Estadísticas de tiempo ciclo
print("\n4. TIEMPO DE CICLO POR CLASIFICACIÓN:")
print("-"*80)
tiempo_stats = df.groupby('etiqueta_clase')['tiempo_ciclo'].agg(['count', 'mean', 'min', 'max', 'std'])
print(tiempo_stats.round(2))

# 5. Correlaciones
print("\n5. CORRELACIONES CON VARIABLES NUMÉRICAS:")
print("-"*80)
# Codificar etiqueta_clase numéricamente para correlación
df_temp = df.copy()
df_temp['etiqueta_clase_num'] = pd.factorize(df_temp['etiqueta_clase'])[0]
correlaciones = df_temp[['tiempo_ciclo', 'costo', 'errores', 'etiqueta_clase_num']].corr()
print(correlaciones['etiqueta_clase_num'].drop('etiqueta_clase_num').round(3))
del df_temp

# 6. Detección de anomalías
print("\n6. DETECCIÓN DE ANOMALÍAS:")
print("-"*80)
anomalias = {
    'tiempo_muy_alto': len(df[df['tiempo_ciclo'] > df['tiempo_ciclo'].quantile(0.95)]),
    'costo_muy_alto': len(df[df['costo'] > df['costo'].quantile(0.95)]),
    'muchos_errores': len(df[df['errores'] >= 3]),
    'registros_urgentes': len(df[df['prioridad'] == 'Urgente'])
}
for tipo, cantidad in anomalias.items():
    pct = (cantidad / len(df)) * 100
    print(f"  {tipo:25} {cantidad:5} registros ({pct:5.1f}%)")


ANÁLISIS DE PATRONES IDENTIFICADOS


1. DISTRIBUCIÓN DE CLASIFICACIÓN (Variable Objetivo):
--------------------------------------------------------------------------------
  Manual           4688 registros ( 93.8%) ██████████████████████████████████████████████
  Automático        250 registros (  5.0%) ██
  Revisión           62 registros (  1.2%) 

2. PATRONES POR TIPO DE PROCESO:
--------------------------------------------------------------------------------
etiqueta_clase  Automático  Manual  Revisión   All
tipo_proceso                                      
Facturación             72    1430        20  1522
Inventario              52     909        11   972
Logística               35     652         9   696
Pagos                   66    1242        17  1325
Soporte                 25     455         5   485
All                    250    4688        62  5000

3. PATRONES POR CANAL:
--------------------------------------------------------------------------------
etiqueta_clase  Auto

## 5️⃣ Clasificación Automática con Reglas

In [5]:
print("AUTOMATIZACIÓN Y CLASIFICACIÓN DE PROCESOS\n")
print("="*80)

# Función de clasificación mejorada
def clasificar_automatico(row):
    """Reglas de negocio para clasificación automática"""
    puntuacion = 0
    razon = []
    
    # Criterios para AUTOMÁTICO
    if row['tiempo_ciclo'] < 20 and row['errores'] == 0 and row['costo'] < 50:
        if row['canal'] in ['Web', 'API']:
            puntuacion += 3
            razon.append("Proceso rápido, sin errores, bajo costo, canal automatizable")
            return 'Automático', puntuacion, razon
    
    # Criterios para REVISIÓN
    if row['errores'] >= 2:
        puntuacion += 2
        razon.append(f"Múltiples errores ({row['errores']})")
    
    if row['prioridad'] == 'Urgente':
        puntuacion += 1
        razon.append("Prioridad Urgente")
    
    if row['costo'] > 500:
        puntuacion += 1
        razon.append("Costo elevado")
    
    if puntuacion >= 2:
        return 'Revisión', puntuacion, razon
    
    # Por defecto MANUAL
    return 'Manual', puntuacion, razon

# Aplicar clasificación
resultados_clasificacion = []
for idx, row in df.iterrows():
    clasificacion, puntuacion, razon = clasificar_automatico(row)
    resultados_clasificacion.append({
        'id': row['id'],
        'clasificacion': clasificacion,
        'puntuacion': puntuacion,
        'razon_principal': razon[0] if razon else 'Proceso estándar'
    })

df_clasificacion = pd.DataFrame(resultados_clasificacion)

# Resumen de clasificación
print("\nRESUMEN DE CLASIFICACIÓN AUTOMÁTICA:\n")
for clase in ['Automático', 'Manual', 'Revisión']:
    count = len(df_clasificacion[df_clasificacion['clasificacion'] == clase])
    pct = (count / len(df_clasificacion)) * 100
    bar = '█' * int(pct / 2)
    print(f"  {clase:15} {count:5} registros ({pct:5.1f}%) {bar}")

# Guardar resultados de clasificación
clasificacion_path = output_dir / "clasificacion_automatica.csv"
df_clasificacion.to_csv(clasificacion_path, index=False)
print(f"\n✓ Resultados guardados: {clasificacion_path}")

# Ejemplos
print(f"\nEJEMPLOS POR CLASIFICACIÓN:\n")
for clase in ['Automático', 'Manual', 'Revisión']:
    print(f"\n{clase.upper()}:")
    ejemplos = df_clasificacion[df_clasificacion['clasificacion'] == clase].head(3)
    for idx, ejemplo in ejemplos.iterrows():
        print(f"  • {ejemplo['id']}: {ejemplo['razon_principal']}")


AUTOMATIZACIÓN Y CLASIFICACIÓN DE PROCESOS


RESUMEN DE CLASIFICACIÓN AUTOMÁTICA:

  Automático        249 registros (  5.0%) ██
  Manual           4313 registros ( 86.3%) ███████████████████████████████████████████
  Revisión          438 registros (  8.8%) ████

✓ Resultados guardados: output_data_processing/clasificacion_automatica.csv

EJEMPLOS POR CLASIFICACIÓN:


AUTOMÁTICO:
  • PROC00014: Proceso rápido, sin errores, bajo costo, canal automatizable
  • PROC00030: Proceso rápido, sin errores, bajo costo, canal automatizable
  • PROC00042: Proceso rápido, sin errores, bajo costo, canal automatizable

MANUAL:
  • PROC00001: Proceso estándar
  • PROC00002: Proceso estándar
  • PROC00003: Proceso estándar

REVISIÓN:
  • PROC00012: Múltiples errores (2)
  • PROC00029: Múltiples errores (2)
  • PROC00039: Múltiples errores (2)


## 6️⃣ Ingesta Unificada desde Múltiples Fuentes

In [6]:
print("INGESTA DE DATOS DESDE MÚLTIPLES FUENTES\n")
print("="*80)

# Dictionary para almacenar datos de diferentes fuentes
datos_ingestas = {}

# 1. Leer desde CSV
print("\n1. Leyendo desde CSV...")
try:
    df_csv = pd.read_csv(csv_path)
    datos_ingestas['CSV'] = {
        'registros': len(df_csv),
        'columnas': list(df_csv.columns),
        'estado': '✓ OK'
    }
    print(f"   ✓ {len(df_csv)} registros cargados desde CSV")
except Exception as e:
    datos_ingestas['CSV'] = {'estado': '✗ Error', 'detalle': str(e)}
    print(f"   ✗ Error: {e}")

# 2. Leer desde Excel
print("\n2. Leyendo desde Excel...")
try:
    df_excel = pd.read_excel(excel_path)
    datos_ingestas['Excel'] = {
        'registros': len(df_excel),
        'columnas': list(df_excel.columns),
        'estado': '✓ OK'
    }
    print(f"   ✓ {len(df_excel)} registros cargados desde Excel")
except Exception as e:
    datos_ingestas['Excel'] = {'estado': '✗ Error', 'detalle': str(e)}
    print(f"   ✗ Error: {e}")

# 3. Leer desde JSON
print("\n3. Leyendo desde JSON...")
try:
    with open(json_path, 'r') as f:
        data_json = json.load(f)
    datos_ingestas['JSON'] = {
        'registros': len(data_json),
        'columnas': list(data_json[0].keys()) if data_json else [],
        'estado': '✓ OK'
    }
    print(f"   ✓ {len(data_json)} registros cargados desde JSON")
except Exception as e:
    datos_ingestas['JSON'] = {'estado': '✗ Error', 'detalle': str(e)}
    print(f"   ✗ Error: {e}")

# 4. Leer desde SQL
print("\n4. Leyendo desde SQL (SQLite)...")
try:
    conn = sqlite3.connect(db_path)
    df_sql = pd.read_sql_query("SELECT * FROM procesos", conn)
    conn.close()
    datos_ingestas['SQL'] = {
        'registros': len(df_sql),
        'columnas': list(df_sql.columns),
        'estado': '✓ OK'
    }
    print(f"   ✓ {len(df_sql)} registros cargados desde SQL")
except Exception as e:
    datos_ingestas['SQL'] = {'estado': '✗ Error', 'detalle': str(e)}
    print(f"   ✗ Error: {e}")

# Resumen de ingesta
print("\n" + "-"*80)
print("\nRESUMEN DE INGESTA:")
print("-"*80 + "\n")
for fuente, info in datos_ingestas.items():
    print(f"{fuente:10} - {info.get('estado', 'Desconocido')} - {info.get('registros', '?')} registros")

# 5. Ingesta unificada (combinar todas las fuentes)
print("\n5. Combinando datos de todas las fuentes...")
df_unificado = pd.concat([df_csv, df_excel, df_sql], ignore_index=True)
df_unificado = df_unificado.drop_duplicates(subset=['id'], keep='first')  # Eliminar duplicados por ID

print(f"\n✓ Dataset unificado:")
print(f"  - Registros originales: {len(df_csv) + len(df_excel) + len(df_sql)}")
print(f"  - Registros después de deduplicación: {len(df_unificado)}")
print(f"  - Duplicados eliminados: {len(df_csv) + len(df_excel) + len(df_sql) - len(df_unificado)}")

# Guardar dataset unificado
unificado_path = output_dir / "dataset_unificado.csv"
df_unificado.to_csv(unificado_path, index=False)
print(f"  - Guardado en: {unificado_path}")


INGESTA DE DATOS DESDE MÚLTIPLES FUENTES


1. Leyendo desde CSV...
   ✓ 5000 registros cargados desde CSV

2. Leyendo desde Excel...
   ✓ 5000 registros cargados desde Excel

3. Leyendo desde JSON...
   ✓ 5000 registros cargados desde JSON

4. Leyendo desde SQL (SQLite)...
   ✓ 5000 registros cargados desde SQL

--------------------------------------------------------------------------------

RESUMEN DE INGESTA:
--------------------------------------------------------------------------------

CSV        - ✓ OK - 5000 registros
Excel      - ✓ OK - 5000 registros
JSON       - ✓ OK - 5000 registros
SQL        - ✓ OK - 5000 registros

5. Combinando datos de todas las fuentes...

✓ Dataset unificado:
  - Registros originales: 15000
  - Registros después de deduplicación: 5000
  - Duplicados eliminados: 10000
  - Guardado en: output_data_processing/dataset_unificado.csv


## 7️⃣ Normalización y Consolidación de Datos

In [7]:
print("NORMALIZACIÓN Y CONSOLIDACIÓN\n")
print("="*80)

# Preparar dataset para modelado
df_modelo = df_unificado.copy()

# 1. Manejo de valores faltantes
print("\n1. Detectando valores faltantes...")
valores_faltantes = df_modelo.isnull().sum()
print(f"\nColumnas con valores faltantes:\n{valores_faltantes[valores_faltantes > 0]}")

# Imputar faltantes
df_modelo['responsable'] = df_modelo['responsable'].fillna('SIN_ASIGNAR')
df_modelo['etiqueta_clase'] = df_modelo['etiqueta_clase'].fillna('Manual')
numeric_cols = df_modelo.select_dtypes(include=[np.number]).columns
for col in numeric_cols:
    df_modelo[col] = df_modelo[col].fillna(df_modelo[col].median())

print(f"✓ Faltantes imputados")

# 2. Normalización de tipos de datos
print("\n2. Estandarizando tipos de datos...")
df_modelo['fecha'] = pd.to_datetime(df_modelo['fecha'], errors='coerce')
df_modelo['tipo_proceso'] = df_modelo['tipo_proceso'].astype('category')
df_modelo['canal'] = df_modelo['canal'].astype('category')
df_modelo['prioridad'] = df_modelo['prioridad'].astype('category')
df_modelo['etiqueta_clase'] = df_modelo['etiqueta_clase'].astype('category')
print("✓ Tipos de datos estandarizados")

# 3. Codificación de variables categóricas
print("\n3. Codificando variables categóricas...")
from sklearn.preprocessing import LabelEncoder
label_encoders = {}
categorical_cols = ['tipo_proceso', 'canal', 'prioridad', 'responsable']

for col in categorical_cols:
    le = LabelEncoder()
    df_modelo[f'{col}_encoded'] = le.fit_transform(df_modelo[col].astype(str))
    label_encoders[col] = le
    print(f"   ✓ {col}: {len(le.classes_)} categorías")

# 4. Normalización de features numéricos
print("\n4. Normalizando features numéricos...")
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
numeric_features = ['tiempo_ciclo', 'costo', 'errores']
df_modelo[['tiempo_ciclo_norm', 'costo_norm', 'errores_norm']] = scaler.fit_transform(df_modelo[numeric_features])
print(f"✓ {len(numeric_features)} features normalizados")

# 5. Ingeniería de features
print("\n5. Creando features derivados...")
df_modelo['mes'] = df_modelo['fecha'].dt.month
df_modelo['dia_semana'] = df_modelo['fecha'].dt.dayofweek
df_modelo['trimestre'] = df_modelo['fecha'].dt.quarter
df_modelo['costo_por_error'] = np.where(df_modelo['errores'] > 0, 
                                         df_modelo['costo'] / df_modelo['errores'], 
                                         0)
df_modelo['tiempo_vs_mediana'] = df_modelo['tiempo_ciclo'] / df_modelo['tiempo_ciclo'].median()
print(f"✓ 5 features derivados creados")

# 6. Dataset consolidado
print("\n" + "-"*80)
print("\nDATASET CONSOLIDADO:")
print("-"*80)
print(f"\nForma: {df_modelo.shape}")
print(f"Columnas: {len(df_modelo.columns)}")
print(f"Tipos de datos:\n{df_modelo.dtypes.value_counts()}")
print(f"\nPrimeras filas del dataset normalizado:")
print(df_modelo[['id', 'fecha', 'tipo_proceso', 'tiempo_ciclo_norm', 'costo_norm', 'etiqueta_clase']].head())

# Guardar dataset normalizado
normalizado_path = output_dir / "dataset_normalizado.csv"
df_modelo.to_csv(normalizado_path, index=False)
print(f"\n✓ Dataset normalizado guardado en: {normalizado_path}")


NORMALIZACIÓN Y CONSOLIDACIÓN


1. Detectando valores faltantes...

Columnas con valores faltantes:
Series([], dtype: int64)
✓ Faltantes imputados

2. Estandarizando tipos de datos...
✓ Tipos de datos estandarizados

3. Codificando variables categóricas...
   ✓ tipo_proceso: 5 categorías
   ✓ canal: 5 categorías
   ✓ prioridad: 4 categorías
   ✓ responsable: 6 categorías

4. Normalizando features numéricos...
✓ 3 features normalizados

5. Creando features derivados...
✓ 5 features derivados creados

--------------------------------------------------------------------------------

DATASET CONSOLIDADO:
--------------------------------------------------------------------------------

Forma: (5000, 22)
Columnas: 22
Tipos de datos:
float64           7
int64             5
int32             3
str               2
datetime64[us]    1
category          1
category          1
category          1
category          1
Name: count, dtype: int64

Primeras filas del dataset normalizado:
          id    

## 8️⃣ Entrenamiento de Modelos de Clasificación

In [14]:
print("ENTRENAMIENTO DE MODELOS\n")
print("="*80)

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from sklearn.utils.class_weight import compute_sample_weight
import pickle

# Preparar datos de entrenamiento
print("\n1. Preparando datos para entrenamiento...")

# Seleccionar features
feature_cols = [col for col in df_modelo.columns 
                if col not in ['id', 'fecha', 'responsable', 'etiqueta_clase', 'tipo_proceso', 
                               'canal', 'prioridad'] 
                and (col.endswith('_encoded') or col.endswith('_norm') or col in ['mes', 'dia_semana', 'trimestre', 'costo_por_error', 'tiempo_vs_mediana'])]

X = df_modelo[feature_cols]
y = df_modelo['etiqueta_clase'].cat.codes  # Codificar la variable objetivo

# Diagnóstico de desbalanceo
class_names = df_modelo['etiqueta_clase'].cat.categories.tolist()
class_distribution = pd.Series(y).value_counts(normalize=True).sort_index()
print("\nDistribución de clases (train target):")
for class_id, class_ratio in class_distribution.items():
    print(f"   {class_names[class_id]:15}: {class_ratio * 100:6.2f}%")

# Dividir datos
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_SEED, stratify=y
)
print(f"✓ Entrenamiento: {len(X_train)} registros")
print(f"✓ Prueba: {len(X_test)} registros")

# 2. Entrenar modelos
print("\n2. Entrenando modelos...")

# Pesos para desbalanceo (útil especialmente en Gradient Boosting)
sample_weights_train = compute_sample_weight(class_weight='balanced', y=y_train)

# Random Forest
print("\n   a) Random Forest...")
rf_model = RandomForestClassifier(
    n_estimators=100,
    max_depth=15,
    random_state=RANDOM_SEED,
    n_jobs=-1,
    class_weight='balanced_subsample'
)
rf_model.fit(X_train, y_train)
rf_pred = rf_model.predict(X_test)
rf_accuracy = accuracy_score(y_test, rf_pred)
print(f"      ✓ Accuracy: {rf_accuracy:.4f}")

# Gradient Boosting
print("\n   b) Gradient Boosting...")
gb_model = GradientBoostingClassifier(
    n_estimators=100,
    max_depth=5,
    learning_rate=0.05,
    random_state=RANDOM_SEED
)
gb_model.fit(X_train, y_train, sample_weight=sample_weights_train)
gb_pred = gb_model.predict(X_test)
gb_accuracy = accuracy_score(y_test, gb_pred)
print(f"      ✓ Accuracy: {gb_accuracy:.4f}")

# 3. Seleccionar mejor modelo
best_model = rf_model if rf_accuracy > gb_accuracy else gb_model
best_model_name = "Random Forest" if rf_accuracy > gb_accuracy else "Gradient Boosting"
best_accuracy = max(rf_accuracy, gb_accuracy)

print(f"\n✓ Mejor modelo: {best_model_name} (Accuracy: {best_accuracy:.4f})")

# 4. Evaluación detallada
print("\n3. Evaluación del modelo...")
print("\n" + "-"*80)
print("MATRIZ DE CONFUSIÓN:")
print("-"*80)
conf_matrix = confusion_matrix(y_test, best_model.predict(X_test))
print(f"\nPredicciones por clase:")
for i, class_name in enumerate(class_names):
    tp = conf_matrix[i, i]
    print(f"   {class_name:15} - TP: {tp:3d} (de {conf_matrix[i].sum():3d} totales)")

print("\n" + "-"*80)
print("REPORTE DE CLASIFICACIÓN:")
print("-"*80)
from sklearn.metrics import precision_score, recall_score, f1_score
print(f"\nAccuracy global: {best_accuracy:.4f}")
print(f"Precision ponderada: {precision_score(y_test, best_model.predict(X_test), average='weighted'):.4f}")
print(f"Recall ponderado: {recall_score(y_test, best_model.predict(X_test), average='weighted'):.4f}")
print(f"F1-Score ponderado: {f1_score(y_test, best_model.predict(X_test), average='weighted'):.4f}")

# 5. Feature importance
print("\n4. Features más relevantes:")
print("-"*80)
feature_importance = pd.DataFrame({
    'feature': feature_cols,
    'importance': best_model.feature_importances_
}).sort_values('importance', ascending=False)

for idx, row in feature_importance.head(10).iterrows():
    print(f"   {row['feature']:25} - {row['importance']:.4f}")

# 6. Guardar modelo
print("\n5. Guardando modelo...")
model_path = output_dir / "clasificador_procesos.pkl"
with open(model_path, 'wb') as f:
    pickle.dump({
        'model': best_model,
        'scaler': scaler,
        'label_encoders': label_encoders,
        'feature_cols': feature_cols,
        'class_names': class_names,
        'accuracy': best_accuracy,
        'class_distribution': class_distribution.to_dict(),
        'random_seed': RANDOM_SEED,
        'pipeline_version': PIPELINE_METADATA['pipeline_version']
    }, f)
print(f"✓ Modelo guardado en: {model_path}")


ENTRENAMIENTO DE MODELOS


1. Preparando datos para entrenamiento...

Distribución de clases (train target):
   Automático     :   5.00%
   Manual         :  93.76%
   Revisión       :   1.24%
✓ Entrenamiento: 4000 registros
✓ Prueba: 1000 registros

2. Entrenando modelos...

   a) Random Forest...
      ✓ Accuracy: 0.9990

   b) Gradient Boosting...
      ✓ Accuracy: 0.9980

✓ Mejor modelo: Random Forest (Accuracy: 0.9990)

3. Evaluación del modelo...

--------------------------------------------------------------------------------
MATRIZ DE CONFUSIÓN:
--------------------------------------------------------------------------------

Predicciones por clase:
   Automático      - TP:  49 (de  50 totales)
   Manual          - TP: 938 (de 938 totales)
   Revisión        - TP:  12 (de  12 totales)

--------------------------------------------------------------------------------
REPORTE DE CLASIFICACIÓN:
--------------------------------------------------------------------------------

Accura

## 9️⃣ Flujo de Automatización e Ingesta de Nuevos Datos

In [18]:
print("FLUJO DE AUTOMATIZACIÓN PARA NUEVOS DATOS\n")
print("="*80)

def safe_label_transform(le, values):
    """Codifica categorías nuevas sin romper la inferencia: desconocidos -> -1."""
    known = set(le.classes_)
    encoded = []
    unknown_count = 0
    for val in values.astype(str):
        if val in known:
            encoded.append(int(le.transform([val])[0]))
        else:
            encoded.append(-1)
            unknown_count += 1
    return np.array(encoded), unknown_count


def procesar_nuevos_datos(nuevo_archivo, archivo_tipo='csv'):
    """Pipeline completo para procesar nuevos datos y hacer predicciones"""

    print(f"\n1. Ingesta de datos ({archivo_tipo.upper()})...")

    # Cargar nuevos datos
    if archivo_tipo == 'csv':
        df_nuevos = pd.read_csv(nuevo_archivo)
    elif archivo_tipo == 'excel':
        df_nuevos = pd.read_excel(nuevo_archivo)
    else:
        raise ValueError(f"Tipo de archivo no soportado: {archivo_tipo}")

    print(f"   ✓ {len(df_nuevos)} nuevos registros cargados")

    # 2. Normalización
    print(f"\n2. Aplicando normalización...")
    df_proc = df_nuevos.copy()

    # Imputar faltantes
    df_proc['responsable'] = df_proc['responsable'].fillna('SIN_ASIGNAR')
    numeric_cols = df_proc.select_dtypes(include=[np.number]).columns
    for col in numeric_cols:
        df_proc[col] = df_proc[col].fillna(df_proc[col].median())

    # Convertir tipos
    df_proc['fecha'] = pd.to_datetime(df_proc['fecha'], errors='coerce')
    df_proc['tipo_proceso'] = df_proc['tipo_proceso'].astype('category')
    df_proc['canal'] = df_proc['canal'].astype('category')
    df_proc['prioridad'] = df_proc['prioridad'].astype('category')

    # Codificar de forma robusta ante categorías no vistas
    unknown_summary = {}
    for col, le in label_encoders.items():
        if col in df_proc.columns:
            encoded, unknown_count = safe_label_transform(le, df_proc[col])
            df_proc[f'{col}_encoded'] = encoded
            unknown_summary[col] = unknown_count

    total_unknowns = sum(unknown_summary.values())
    if total_unknowns > 0:
        print(f"   ⚠ Categorías no vistas detectadas: {total_unknowns}")
        for col_name, col_unknowns in unknown_summary.items():
            if col_unknowns > 0:
                print(f"      - {col_name}: {col_unknowns}")

    # Normalizar numéricos
    df_proc[['tiempo_ciclo_norm', 'costo_norm', 'errores_norm']] = scaler.transform(
        df_proc[['tiempo_ciclo', 'costo', 'errores']]
    )

    # Features derivados
    df_proc['mes'] = df_proc['fecha'].dt.month
    df_proc['dia_semana'] = df_proc['fecha'].dt.dayofweek
    df_proc['trimestre'] = df_proc['fecha'].dt.quarter
    df_proc['costo_por_error'] = np.where(
        df_proc['errores'] > 0,
        df_proc['costo'] / df_proc['errores'],
        0
    )
    df_proc['tiempo_vs_mediana'] = df_proc['tiempo_ciclo'] / df_proc['tiempo_ciclo'].median()

    print(f"   ✓ Datos normalizados")

    # 3. Predicciones
    print(f"\n3. Generando predicciones con el modelo entrenado...")
    X_nuevos = df_proc[feature_cols]
    predicciones = best_model.predict(X_nuevos)
    probabilidades = best_model.predict_proba(X_nuevos)

    df_proc['prediccion_clase'] = [class_names[pred] for pred in predicciones]
    df_proc['confianza_prediccion'] = probabilidades.max(axis=1)

    print(f"   ✓ {len(predicciones)} predicciones generadas")

    # 4. Aplicar reglas de negocio y automatización
    print(f"\n4. Aplicando reglas de negocio y acciones automáticas...")
    print(
        f"   · Umbrales: BATCH_CONFIDENCE={AUTOMATION_THRESHOLDS['BATCH_CONFIDENCE']} | "
        f"REVIEW_NOTIFY_CONFIDENCE={AUTOMATION_THRESHOLDS['REVIEW_NOTIFY_CONFIDENCE']}"
    )

    df_proc['accion_automatica'] = 'REVISAR'
    df_proc['prioridad_automatica'] = 'Media'
    df_proc['asignado_a'] = 'Equipo_General'

    # Si hay categorías no vistas, forzar revisión humana
    if total_unknowns > 0:
        unseen_mask = False
        for col in ['tipo_proceso_encoded', 'canal_encoded', 'prioridad_encoded', 'responsable_encoded']:
            if col in df_proc.columns:
                unseen_mask = unseen_mask | (df_proc[col] == -1)
        df_proc.loc[unseen_mask, 'accion_automatica'] = 'REVISAR_DATOS_NUEVOS'
        df_proc.loc[unseen_mask, 'prioridad_automatica'] = 'Alta'
        df_proc.loc[unseen_mask, 'asignado_a'] = 'Data_Steward'

    # Reglas automáticas
    for idx, row in df_proc.iterrows():
        # Automáticas de baja prioridad
        if (
            row['confianza_prediccion'] > AUTOMATION_THRESHOLDS['BATCH_CONFIDENCE']
            and row['prediccion_clase'] == 'Automático'
        ):
            df_proc.at[idx, 'accion_automatica'] = 'PROCESAR_BATCH'
            df_proc.at[idx, 'prioridad_automatica'] = 'Baja'
            df_proc.at[idx, 'asignado_a'] = 'Robot_Procesos'

        # Revisión con alta confianza en clase de riesgo
        elif (
            row['prediccion_clase'] == 'Revisión'
            and row['confianza_prediccion'] > AUTOMATION_THRESHOLDS['REVIEW_NOTIFY_CONFIDENCE']
        ):
            df_proc.at[idx, 'accion_automatica'] = 'NOTIFICAR'
            df_proc.at[idx, 'prioridad_automatica'] = 'Alta'
            df_proc.at[idx, 'asignado_a'] = 'Equipo_Senior'

    print(f"   ✓ Acciones automáticas asignadas")
    print(f"      - PROCESAR_BATCH: {(df_proc['accion_automatica'] == 'PROCESAR_BATCH').sum()}")
    print(f"      - NOTIFICAR: {(df_proc['accion_automatica'] == 'NOTIFICAR').sum()}")
    print(f"      - REVISAR: {(df_proc['accion_automatica'] == 'REVISAR').sum()}")
    print(f"      - REVISAR_DATOS_NUEVOS: {(df_proc['accion_automatica'] == 'REVISAR_DATOS_NUEVOS').sum()}")

    # 5. Guardar resultados
    print(f"\n5. Guardando resultados...")
    resultado_path = output_dir / f"predicciones_{pd.Timestamp.now().strftime('%Y%m%d_%H%M%S')}.csv"
    output_cols = [
        'id', 'fecha', 'tipo_proceso', 'canal', 'tiempo_ciclo', 'costo',
        'errores', 'prediccion_clase', 'confianza_prediccion',
        'accion_automatica', 'prioridad_automatica', 'asignado_a'
    ]
    df_proc[output_cols].to_csv(resultado_path, index=False)
    print(f"   ✓ Resultados guardados en: {resultado_path}")

    return df_proc[output_cols]

# Demostración con datos de prueba (usando parte del dataset de test)
print("\n" + "-"*80)
print("DEMOSTRACIÓN 1: Simulando ingesta de 50 nuevos registros")
print("-"*80)

# Generar archivo de prueba
nuevos_datos_sample = df_unificado.sample(n=50, random_state=99).copy()
nuevos_datos_sample['id'] = range(10001, 10051)  # IDs nuevos
test_file = output_dir / "nuevos_procesos_muestra.csv"
nuevos_datos_sample.to_csv(test_file, index=False)

# Procesar nuevos datos
resultados = procesar_nuevos_datos(test_file, archivo_tipo='csv')
print(f"\nMuestra de predicciones:")
print(resultados.head(10))

print("\n" + "-"*80)
print("DEMOSTRACIÓN 2: Forzando categorías no vistas")
print("-"*80)

# Crear muestra con valores no vistos para validar ruta Data Steward
muestra_no_vista = df_unificado.sample(n=20, random_state=123).copy()
muestra_no_vista['id'] = range(20001, 20021)
muestra_no_vista.loc[muestra_no_vista.index[:7], 'canal'] = 'WhatsApp'
muestra_no_vista.loc[muestra_no_vista.index[7:14], 'tipo_proceso'] = 'Fraude'
muestra_no_vista.loc[muestra_no_vista.index[14:17], 'prioridad'] = 'Crítica'
muestra_no_vista.loc[muestra_no_vista.index[17:20], 'responsable'] = 'BOT_X'

no_vistos_file = output_dir / "nuevos_procesos_no_vistos.csv"
muestra_no_vista.to_csv(no_vistos_file, index=False)

resultados_no_vistos = procesar_nuevos_datos(no_vistos_file, archivo_tipo='csv')
print("\nResumen de acciones para categorías no vistas:")
print(resultados_no_vistos['accion_automatica'].value_counts())


FLUJO DE AUTOMATIZACIÓN PARA NUEVOS DATOS


--------------------------------------------------------------------------------
DEMOSTRACIÓN 1: Simulando ingesta de 50 nuevos registros
--------------------------------------------------------------------------------

1. Ingesta de datos (CSV)...
   ✓ 50 nuevos registros cargados

2. Aplicando normalización...
   ✓ Datos normalizados

3. Generando predicciones con el modelo entrenado...
   ✓ 50 predicciones generadas

4. Aplicando reglas de negocio y acciones automáticas...
   · Umbrales: BATCH_CONFIDENCE=0.85 | REVIEW_NOTIFY_CONFIDENCE=0.7
   ✓ Acciones automáticas asignadas
      - PROCESAR_BATCH: 2
      - NOTIFICAR: 1
      - REVISAR: 47
      - REVISAR_DATOS_NUEVOS: 0

5. Guardando resultados...
   ✓ Resultados guardados en: output_data_processing/predicciones_20260321_204312.csv

Muestra de predicciones:
      id               fecha tipo_proceso   canal  tiempo_ciclo   costo  \
0  10001 2024-01-03 10:27:00   Inventario  Manual        

## 🔟 Evaluación, Auditabilidad y Exportación Final

In [19]:
print("EVALUACIÓN, AUDITABILIDAD Y EXPORTACIÓN FINAL\n")
print("="*80)

# 1. Reporte de Evaluación del Modelo
print("\n1. REPORTE COMPLETO DEL MODELO")
print("-"*80)

reporte_evaluacion = {
    'Métrica': [],
    'Valor': [],
    'Descripción': []
}

# Confiabilidad del modelo
y_pred_final = best_model.predict(X_test)
from sklearn.metrics import precision_score, recall_score, f1_score, roc_auc_score

for i, class_name in enumerate(class_names):
    y_binary = (y_test == i).astype(int)
    try:
        auc = roc_auc_score(y_binary, best_model.predict_proba(X_test)[:, i])
        reporte_evaluacion['Métrica'].append(f'AUC-ROC ({class_name})')
        reporte_evaluacion['Valor'].append(f'{auc:.4f}')
        reporte_evaluacion['Descripción'].append('Área bajo la curva ROC')
    except Exception:
        pass

# Métricas globales
acc = accuracy_score(y_test, y_pred_final)
prec = precision_score(y_test, y_pred_final, average='weighted', zero_division=0)
rec = recall_score(y_test, y_pred_final, average='weighted')
f1 = f1_score(y_test, y_pred_final, average='weighted')

reporte_evaluacion['Métrica'].extend(['Accuracy', 'Precision', 'Recall', 'F1-Score'])
reporte_evaluacion['Valor'].extend([f'{acc:.4f}', f'{prec:.4f}', f'{rec:.4f}', f'{f1:.4f}'])
reporte_evaluacion['Descripción'].extend(['Exactitud global', 'Precisión ponderada', 'Cobertura ponderada', 'Media armónica'])

df_reporte = pd.DataFrame(reporte_evaluacion)
print(df_reporte.to_string(index=False))

# Usar el DataFrame de salida generado en la sección 9
resultados_df = resultados.copy()

# 2. Log de Auditabilidad
print("\n\n2. LOG DE AUDITABILIDAD")
print("-"*80)

audit_log = {
    'timestamp': pd.Timestamp.now(),
    'evento': 'Pipeline Completado',
    'registros_procesados': len(df_modelo),
    'registros_entrenamiento': len(X_train),
    'registros_prueba': len(X_test),
    'modelo_utilizado': best_model_name,
    'accuracy': best_accuracy,
    'features_utilizados': len(feature_cols),
    'versión_pipeline': PIPELINE_METADATA['pipeline_version'],
    'random_seed': PIPELINE_METADATA['random_seed']
}

print("\nInfo del Pipeline:")
for key, value in audit_log.items():
    if key != 'timestamp':
        print(f"  • {key}: {value}")
    else:
        print(f"  • {key}: {value.strftime('%Y-%m-%d %H:%M:%S')}")

# 3. Métricas por Categoría
print("\n\n3. DISTRIBUCIÓN FINAL DE CLASIFICACIONES")
print("-"*80)

dist_entreno = df_unificado['etiqueta_clase'].value_counts().sort_values(ascending=False)
dist_prediccion = resultados_df['prediccion_clase'].value_counts().sort_values(ascending=False)

print("\nDataset Original:")
for clase, count in dist_entreno.items():
    pct = (count / len(df_unificado)) * 100
    print(f"  • {clase:20} - {count:5d} registros ({pct:5.2f}%)")

print("\nPredicciones en Nuevos Datos:")
for clase, count in dist_prediccion.items():
    pct = (count / len(resultados_df)) * 100
    print(f"  • {clase:20} - {count:5d} registros ({pct:5.2f}%)")

# 4. Análisis de Confianza
print("\n\n4. ANÁLISIS DE CONFIANZA DE PREDICCIONES")
print("-"*80)

confianza_stats = {
    'Métrica': ['Mínima', 'Promedio', 'Mediana', 'Máxima', 'Desv. Est.'],
    'Confianza': [
        f'{resultados_df["confianza_prediccion"].min():.4f}',
        f'{resultados_df["confianza_prediccion"].mean():.4f}',
        f'{resultados_df["confianza_prediccion"].median():.4f}',
        f'{resultados_df["confianza_prediccion"].max():.4f}',
        f'{resultados_df["confianza_prediccion"].std():.4f}'
    ]
}

for metric, conf in zip(confianza_stats['Métrica'], confianza_stats['Confianza']):
    print(f"  • {metric:15} - {conf}")

# Predicciones de alta confianza
high_conf = (resultados_df['confianza_prediccion'] > 0.85).sum()
print(f"\n  • Predicciones con confianza > 85%: {high_conf} ({(high_conf/len(resultados_df)*100):.2f}%)")

# 5. Exportación de Resultados
print("\n\n5. EXPORTACIÓN DE RESULTADOS FINALES")
print("-"*80)

# CSV con predicciones
export_df = resultados_df[['id', 'fecha', 'tipo_proceso', 'canal', 'tiempo_ciclo', 'costo', 'errores',
                           'prediccion_clase', 'confianza_prediccion', 'accion_automatica',
                           'prioridad_automatica', 'asignado_a']].copy()

csv_export_path = output_dir / "analisis_final_predicciones.csv"
export_df.to_csv(csv_export_path, index=False)
print(f"\n✓ CSV: {csv_export_path.name}")

# Excel con múltiples hojas
excel_export_path = output_dir / "analisis_final_completo.xlsx"
with pd.ExcelWriter(excel_export_path, engine='openpyxl') as writer:
    export_df.to_excel(writer, sheet_name='Predicciones', index=False)
    df_reporte.to_excel(writer, sheet_name='Métricas', index=False)

    # Estadísticas por categoría
    stats_by_class = resultados_df.groupby('prediccion_clase')[['tiempo_ciclo', 'costo', 'errores', 'confianza_prediccion']].agg(['mean', 'min', 'max', 'std'])
    stats_by_class.to_excel(writer, sheet_name='Estadísticas')
print(f"✓ Excel: {excel_export_path.name}")

# JSON con metadatos
json_export = {
    'metadata': {
        'versión': PIPELINE_METADATA['pipeline_version'],
        'fecha_ejecución': str(pd.Timestamp.now()),
        'total_registros': len(resultados_df),
        'modelo': best_model_name,
        'accuracy': float(best_accuracy),
        'confianza_promedio': float(resultados_df['confianza_prediccion'].mean()),
        'random_seed': PIPELINE_METADATA['random_seed'],
        'class_distribution_train': {str(k): float(v) for k, v in class_distribution.items()}
    },
    'distribución_predicciones': dist_prediccion.to_dict(),
    'acciones_automáticas': {
        'PROCESAR_BATCH': int((resultados_df['accion_automatica'] == 'PROCESAR_BATCH').sum()),
        'NOTIFICAR': int((resultados_df['accion_automatica'] == 'NOTIFICAR').sum()),
        'REVISAR': int((resultados_df['accion_automatica'] == 'REVISAR').sum()),
        'REVISAR_DATOS_NUEVOS': int((resultados_df['accion_automatica'] == 'REVISAR_DATOS_NUEVOS').sum())
    }
}

json_export_path = output_dir / "analisis_metadatos.json"
with open(json_export_path, 'w') as f:
    json.dump(json_export, f, indent=2)
print(f"✓ JSON: {json_export_path.name}")

# 6. Resumen Ejecutivo
print("\n\n" + "="*80)
print("RESUMEN EJECUTIVO FINAL")
print("="*80)

print(f"""
📊 SOLUCIÓN IMPLEMENTADA:
   ✓ Ingesta unificada de 4 fuentes: CSV, Excel, SQL, JSON
   ✓ 5000+ registros procesados con 10 dimensiones de análisis
   ✓ Modelo de clasificación automática: {best_model_name}
   ✓ Confiabilidad: {best_accuracy:.2%}
   ✓ Semilla de reproducibilidad: {PIPELINE_METADATA['random_seed']}

🤖 AUTOMATIZACIÓN CONSEGUIDA:
   • {(resultados_df['accion_automatica'] == 'PROCESAR_BATCH').sum()} procesos en batch automático
   • {(resultados_df['accion_automatica'] == 'NOTIFICAR').sum()} requieren notificación a equipos
   • {(resultados_df['accion_automatica'] == 'REVISAR_DATOS_NUEVOS').sum()} con categorías no vistas
   • Ahorro potencial: ~{(resultados_df['accion_automatica'] == 'PROCESAR_BATCH').sum()} revisiones manuales

📈 PATRONES IDENTIFICADOS:
   • {(resultados_df['prediccion_clase'] == 'Manual').sum()} procesos clasificados como Manual
   • {(resultados_df['prediccion_clase'] == 'Revisión').sum()} procesos en Revisión
   • Tiempo promedio: {resultados_df['tiempo_ciclo'].mean():.1f} horas
   • Costo promedio: ${resultados_df['costo'].mean():.2f}

🎯 RESULTADOS:
   ✓ {len(list(output_dir.glob('*')))} archivos generados en {output_dir}
   ✓ Modelos entrenados y listos para producción
   ✓ Pipeline automatizado y reproducible
   ✓ Trazabilidad completa de decisiones
""")

print("="*80)
print("✓ ANÁLISIS COMPLETO - Pipeline finalizado exitosamente")
print("="*80)


EVALUACIÓN, AUDITABILIDAD Y EXPORTACIÓN FINAL


1. REPORTE COMPLETO DEL MODELO
--------------------------------------------------------------------------------
             Métrica  Valor            Descripción
AUC-ROC (Automático) 0.9995 Área bajo la curva ROC
    AUC-ROC (Manual) 0.9996 Área bajo la curva ROC
  AUC-ROC (Revisión) 1.0000 Área bajo la curva ROC
            Accuracy 0.9990       Exactitud global
           Precision 0.9990    Precisión ponderada
              Recall 0.9990    Cobertura ponderada
            F1-Score 0.9990         Media armónica


2. LOG DE AUDITABILIDAD
--------------------------------------------------------------------------------

Info del Pipeline:
  • timestamp: 2026-03-21 20:43:15
  • evento: Pipeline Completado
  • registros_procesados: 5000
  • registros_entrenamiento: 4000
  • registros_prueba: 1000
  • modelo_utilizado: Random Forest
  • accuracy: 0.999
  • features_utilizados: 12
  • versión_pipeline: 1.2
  • random_seed: 42


3. DISTRIBUCIÓ